# Модуль 7. Разбор кейсов: Кураторство активности и происхождение данных

**Модуль 7. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** activities.csv — восемь искусственных записей; esol.csv — отдельный настоящий открытый корпус.

Отделить сопоставимые точные наблюдения от других endpoint, границ и неизвестных единиц.

**Workflow:** Схема → endpoint/мишень/ассей → единицы/отношение → журнал → snapshot hash.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab07_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%207_Database.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [1]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab07/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


Python: /home/miakhlov/course-notebooks/.venv/bin/python


## 1. Сначала определяем сопоставимость

Активность относится к мишени, endpoint и условиям ассая. Учебная политика допускает точный IC50 для T1/A1; Ki и другой assay не становятся эквивалентными после пересчёта единиц. Читаем исходные строки как журнал наблюдений, а не как готовый target для модели.

**Перед запуском:** Прочитайте условия консервативной политики T1/A1/IC50/=.

In [2]:
from chemistry import curate_activity
import hashlib
with (DATA/"activities.csv").open(encoding="utf-8") as handle:
    records=[curate_activity(row) for row in csv.DictReader(handle)]


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Сначала определяем сопоставимость — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. Причина исключения — часть результата

Функция curate_activity возвращает исходные поля, status и reason. Точные положительные концентрации переводятся в pIC50, а границы не подставляются как точные числа. При монотонно убывающем −log10 знак неравенства меняется. Проверяем число всех строк и происхождение включённых записей.

**Перед запуском:** Преобразуйте только положительные концентрации с известными единицами.

In [3]:
table=pd.DataFrame(records);display(table)
included=[r for r in records if r["status"]=="included"]
assert [r["record_id"] for r in included]==["R01","R02"]
assert all(math.isclose(r["pIC50"],7) for r in included)
assert len(records)==8
table.to_csv(OUT/"activity_audit.csv",index=False)


,record_id,molecule_id,target_id,assay_id,endpoint,value,unit,relation,status,pIC50,reason
0,R01,M01,T1,A1,IC50,100,nM,=,included,7.0,teaching rules satisfied
1,R02,M01,T1,A1,IC50,0.1,uM,=,included,7.0,teaching rules satisfied
2,R03,M02,T1,A1,IC50,10,uM,>,excluded,NaN,relation
3,R04,M03,T1,A1,Ki,50,nM,=,excluded,NaN,endpoint
4,R05,M04,T2,A1,IC50,20,nM,=,excluded,NaN,target_id
5,R06,M05,T1,A2,IC50,30,nM,=,excluded,NaN,assay_id
6,R07,M06,T1,A1,IC50,0,nM,=,excluded,NaN,value_or_unit
7,R08,M07,T1,A1,IC50,100,unknown,=,excluded,NaN,value_or_unit


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Причина исключения — часть результата — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Snapshot и измеренный target

ESOL — отдельный реальный набор; учебная activity-таблица искусственная. Название колонки измеренного logS важно: готовый прогноз ESOL нельзя использовать как экспериментальную цель. Hash идентифицирует полученный файл. Для API дополнительно нужны запрос, дата, pagination, ошибки и сохранённый snapshot, а не только URL.

**Перед запуском:** Сохраните восемь исходных строк и причины шести исключений, отдельно прочитайте паспорт ESOL.

In [4]:
esol=pd.read_csv(DATA/"esol.csv")
print("ESOL отдельно:",len(esol),"строк; measured target присутствует")
assert "measured log solubility in mols per litre" in esol
artifact={"included":2,"excluded":6,"esol_rows":len(esol),"esol_sha256":hashlib.sha256((DATA/"esol.csv").read_bytes()).hexdigest()}


ESOL отдельно: 1128 строк; measured target присутствует


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Snapshot и измеренный target — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

Включены R01/R02 с pIC50=7; в журнале все 8 строк; остальные причины различимы.

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [5]:
record={'module':7,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

/home/miakhlov/course-notebooks/Labs/results/lab07/worked/benchmark.json


## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [Карта чтения](../../Literature/READING_MAP.md).

Перейдите к [lab07_practice.ipynb](lab07_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.